# Lab: Materialmodellierung auf der Kontinuumsskala

**Modul: Materialmodellierung | Helmut-Schmidt-Universität Hamburg**

---

In diesem Lab simulieren Sie zwei wichtige Klassen von Kontinuumsmodellen aus der Vorlesung:

1. **Die Poisson-Gleichung** — angewendet auf die Durchbiegung einer vorgespannten elastischen Membran unter einer Querlast (Vorlesungen 3 & 4)
2. **Das Cahn-Hilliard-Phasenfeldmodell** — Simulation der spinodalen Entmischung, der spontanen Entmischung einer binären Legierung (Vorlesung 3)

Beide Probleme werden mit der **Finite-Elemente-Methode (FEM)** über die [FEniCS](https://fenicsproject.org/)-Bibliothek gelöst.
FEniCS nimmt die mathematische *schwache Form* Ihrer PDE und assembliert automatisch das algebraische System, das ein Computer löst.

**Lernziele**
- Aufsetzen und Lösen eines FEM-Problems in FEniCS ausgehend von einer PDE-Beschreibung
- Verstehen, wie Randbedingungen und Lastverteilungen die Poisson-Lösung formen
- Durchführen einer zeitabhängigen Phasenfeldsimulation und Beobachten der Gefügebildung
- Erklären, wie der Grenzflächenenergiekoeffizient $\kappa$ die Längenskala der Phasenseparation steuert

## Erste Schritte

Arbeiten Sie die Zellen **von oben nach unten** durch.
Jeder Abschnitt beginnt mit der **Physik** des Problems und wird von der FEniCS-**Implementierung** gefolgt.

- Führen Sie eine Zelle mit **Shift + Enter** aus
- Zellen mit `# AKTION` enthalten Parameter, die Sie **ändern und ausprobieren** sollen
- Fügen Sie eigene Notizen durch Einfügen von Markdown-Zellen hinzu (*Insert → Insert Cell Below*, dann Typ auf Markdown ändern)
- FEniCS kann beim ersten Mal einige Initialisierungszeilen ausgeben — das ist normal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

# FEniCS — alle FEM-Funktionalität befindet sich in dolfin
from dolfin import *

# Muss NACH 'from dolfin import *' kommen: dolfin exportiert ein eigenes Modul namens
# 'timer' (dolfin.common.timer), das diesen Import sonst überschatten würde.
import time as timer

# FEniCS-Fortschrittsmeldungen unterdrücken (20 für INFO, 10 für DEBUG)
set_log_level(30)

print("NumPy  version:", np.__version__)

---
## Teil 1: Die Poisson-Gleichung — Elastische Membran

### Physikalisches Problem

Betrachten Sie eine dünne elastische Membran, die über einen starren quadratischen Rahmen gespannt ist.
Die Membran ist vorgespannt (wie ein Trommelfell) und unterliegt einer Querlast $f(x,y)$
(Kraft pro Flächeneinheit, z. B. durch eine Druckdifferenz).
Im Gleichgewicht erfüllt die Auslenkung $u(x,y)$ aus der Ruhelage die **Poisson-Gleichung**:

$$
-\nabla^2 u(x,y) = g(x,y) \quad \text{in } \Omega = [0,1]^2
$$

wobei $g = f/T$ die durch die Membranspannung $T$ normierte Last ist und
$\nabla^2 = \partial^2/\partial x^2 + \partial^2/\partial y^2$ der Laplace-Operator ist.

Die Membran ist an allen vier Rändern eingespannt, was **Dirichlet-Randbedingungen** ergibt:

$$
u = 0 \quad \text{auf } \partial\Omega
$$

> **Bezug zur Vorlesung:** Dieselbe Gleichung beschreibt Wärmeleitung, Diffusion, elektrisches
> Potential und druckgetriebene inkompressible Strömung — nur die physikalische Interpretation von
> $u$ und $g$ ändert sich.

### Variationelle (schwache) Form

FEniCS arbeitet mit der **schwachen Form** der PDE.
Multiplikation mit einer Testfunktion $v$ (mit $v = 0$ auf $\partial\Omega$) und Integration per partes:

$$
\underbrace{\int_\Omega \nabla u \cdot \nabla v \, d\mathbf{r}}_{a(u,\,v)}
= \underbrace{\int_\Omega g \, v \, d\mathbf{r}}_{L(v)}
$$

Der Randterm verschwindet, da $v = 0$ auf $\partial\Omega$.
FEniCS löst $a(u,v) = L(v)$ für alle $v$, indem $u$ und $v$ in
stückweise linearen Basisfunktionen auf den Dreiecken des Gitters entwickelt werden.

In [ ]:
# ================================================================
#  TEIL 1 PARAMETER  — ändern Sie diese, um das Problem zu erkunden
# ================================================================

# Gitterauflösung: Anzahl der Dreieckelemente entlang jeder Kante
nx = 32    # AKTION: probieren Sie 8, 16, 32, 64
ny = 32

# Normierte Lastverteilung g(x,y)
# FEniCS-Ausdrücke verwenden C++-Syntax: 'x[0]' = x-Koordinate, 'x[1]' = y-Koordinate
#
# Option A — gleichmäßige Last:
load_str = "1.0"
#
# Option B — Gaußförmig (zentrale Punktlast), gleiche Spitzenamplitude:
# load_str = "exp(-50*(pow(x[0]-0.5,2) + pow(x[1]-0.5,2)))"
#
# Option C — zwei symmetrische Punktlasten:
# load_str = "exp(-200*(pow(x[0]-0.3,2)+pow(x[1]-0.5,2))) + exp(-200*(pow(x[0]-0.7,2)+pow(x[1]-0.5,2)))"
#
# Option D — linearer Anstieg (asymmetrisch):
# load_str = "2.0*x[0]"
#
# AKTION: kommentieren Sie genau EINE Option oben aus

print(f"Mesh: {nx}x{ny}  |  last: {load_str[:60]}")

In [ ]:
# --- Gitter und Funktionsraum erstellen ---
# UnitSquareMesh erzeugt ein trianguliertes Gitter auf [0,1] x [0,1]
mesh = UnitSquareMesh(nx, ny)

# P1-Lagrange-Elemente: stückweise lineare Formfunktionen
# Jeder Knoten des Gitters trägt einen Freiheitsgrad (DOF)
V = FunctionSpace(mesh, "Lagrange", 1)

print(f"Gitter:  {mesh.num_vertices()} Knoten, {mesh.num_cells()} Dreiecke")
print(f"DOFs:  {V.dim()}")

# --- Randbedingung: u = 0 überall auf dem Rand ---
bc = DirichletBC(V, Constant(0.0), "on_boundary")

# --- Das Variationsproblem definieren ---
u_p = TrialFunction(V)   # unbekannte Auslenkung  (Ansatzfunktion)
v_p = TestFunction(V)    # beliebige Testfunktion

# Lastverteilung
g = Expression(load_str, degree=2)

# Bilinearform:  a(u,v) = Integral von grad(u).grad(v)
a = inner(grad(u_p), grad(v_p)) * dx

# Linearform:    L(v)   = Integral von g*v
L_form = g * v_p * dx

# --- Lösen: finde u so dass a(u,v) = L(v) für alle v ---
u_sol = Function(V)
solve(a == L_form, u_sol, bc)

print(f"\nGelöst!  u_max = {u_sol.vector().max():.6f}")

In [ ]:
# --- Auslenkungsfeld visualisieren ---
# Die FEniCS-Lösung wird auf einem regelmäßigen Gitter gesampelt für saubere matplotlib-Plots
N_vis = 80
x_pts = np.linspace(0.005, 0.995, N_vis)   # leicht innerhalb des Rands
y_pts = np.linspace(0.005, 0.995, N_vis)

Z = np.zeros((N_vis, N_vis))
for i, yj in enumerate(y_pts):
    for j, xi in enumerate(x_pts):
        Z[i, j] = u_sol(xi, yj)   # FEniCS-Punktauswertung

X, Y = np.meshgrid(x_pts, y_pts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# 2D-Farbkarte
im = axes[0].contourf(X, Y, Z, levels=30, cmap='plasma')
plt.colorbar(im, ax=axes[0], label="Auslenkung u")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[0].set_title("Membranauslenkung u(x,y)")
axes[0].set_aspect("equal")

# Mittellinienprofil entlang y = 0.5
profile = [u_sol(xi, 0.5) for xi in x_pts]
axes[1].plot(x_pts, profile, "b-", linewidth=2, label="FEM (y = 0.5)")
axes[1].set_xlabel("x")
axes[1].set_ylabel("u(x, 0.5)")
axes[1].set_title("Mittellinienprofil der Auslenkung")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Maximale Auslenkung: {u_sol.vector().max():.6f}")

### Gitterkonvergenz

Eine zentrale Frage in der FEM: *Wie fein muss das Gitter sein?*
Wenn man das Gitter immer weiter verfeinert und die Lösung sich nicht mehr ändert, hat sie **konvergiert**.
Die folgende Zelle löst den Fall mit gleichmäßiger Last auf progressiv feineren Gittern und verfolgt
die maximale Durchbiegung.

In [ ]:
print("Konvergenzstudie wird ausgeführt (gleichmäßige Last g=1)...")
mesh_sizes   = [4, 8, 16, 32, 64]
max_defl     = []
n_dofs_list  = []

for n in mesh_sizes:
    m   = UnitSquareMesh(n, n)
    V_c = FunctionSpace(m, "Lagrange", 1)
    bc_c = DirichletBC(V_c, Constant(0.0), "on_boundary")
    u_c  = TrialFunction(V_c)
    v_c  = TestFunction(V_c)
    a_c  = inner(grad(u_c), grad(v_c)) * dx
    L_c  = Constant(1.0) * v_c * dx
    u_s  = Function(V_c)
    solve(a_c == L_c, u_s, bc_c)
    max_defl.append(u_s.vector().max())
    n_dofs_list.append(V_c.dim())
    print(f"  n={n:3d}  DOFs={V_c.dim():6d}  u_max={u_s.vector().max():.6f}")

plt.figure(figsize=(7, 4))
plt.semilogx(n_dofs_list, max_defl, "o-", color="steelblue", linewidth=2, markersize=8)
plt.xlabel("Anzahl der DOFs")
plt.ylabel("Maximale Auslenkung")
plt.title("Gitterkonvergenz (gleichmäßige Last g = 1)")
plt.grid(True, alpha=0.3, which="both")
plt.tight_layout()
plt.show()
print(f"\nKonvergierter Wert:  u_max = {max_defl[-1]:.5f}")

### Aufgabe 1: Das Membranproblem erkunden

Kehren Sie zur **Parameter**-Zelle oben zurück und probieren Sie die folgenden Experimente aus.
Führen Sie für jede Änderung die Zellen Parameter, Lösen und Visualisieren erneut aus.

**1.1 — Lastverteilung**
- Wechseln Sie zu Option B (Gaußsche Zentrallast). Wie ändert sich das Durchbiegungsprofil im Vergleich
  zur gleichmäßigen Last? Wo liegt die maximale Durchbiegung?
- Welcher Fall erzeugt bei gleicher Spitzenamplitude eine größere *maximale* Durchbiegung — gleichmäßig oder Gaußförmig?

**1.2 — Asymmetrische Belastung**
- Probieren Sie Option D (linearer Anstieg). Ist die Lösung symmetrisch? Warum oder warum nicht?
- Beschreiben Sie das Mittellinienprofil: Wo liegt das Maximum?

**1.3 — Zwei Punktlasten**
- Probieren Sie Option C. Wie viele Durchbiegungsmaxima sehen Sie? Was passiert im Zentrum?

**1.4 — Konvergenz**
- Aus dem Konvergenzplot: Wie viele DOFs werden ungefähr benötigt, bevor die Lösung
  als konvergiert gilt (Änderung kleiner als 1 %)?

> *Notieren Sie Ihre Beobachtungen, indem Sie unterhalb jedes Experiments eine Markdown-Zelle einfügen.*

---
## Teil 2: Phasenfeldmodellierung — Spinodale Entmischung

### Physikalischer Hintergrund

Wenn eine binäre Legierung (z. B. Sn-Pb-Lot) schnell in den **spinodalen Bereich** des
Phasendiagramms abgekühlt wird, wird sie thermodynamisch instabil: Jede kleine Zusammensetzungsfluktuation
verstärkt sich spontan und treibt das System ohne Nukleation zur Phasenseparation.
Dies ist die **spinodale Entmischung**.

Das **Cahn-Hilliard-Modell** beschreibt dies mit einem skalaren Ordnungsparameter $n(\mathbf{r},t)$
— der lokalen Zusammensetzung:
- $n = -1$: reine Komponente A
- $n = +1$: reine Komponente B

### Freienergifunktional

Die totale freie Energie (aus Vorlesung 3) ist:

$$
E[n,\nabla n] = \int_\Omega
\underbrace{G(n)}_{\text{Volumenenergie}}
+ \underbrace{\frac{\kappa}{2}|\nabla n|^2}_{\text{Grenzflächenenergie}}
\, d\mathbf{r}
$$

Die **Volumenenergie** ist ein Doppelmuldenpotential mit Minima bei $n = \pm 1$:

$$
G(n) = (1 - n^2)^2
$$

Jede Zusammensetzung zwischen den beiden Phasen ist im Volumen energetisch ungünstig.
Der **Gradiententerm** bestraft scharfe Grenzflächen; der Parameter $\kappa$ legt die Grenzflächenbreite fest.

### Bewegungsgleichung

Die Massenerhaltung liefert die **Cahn-Hilliard-Gleichung**:

$$
\frac{\partial n}{\partial t} = \nabla^2 \tilde{\mu}, \qquad
\tilde{\mu} = \frac{\partial G}{\partial n} - \kappa\nabla^2 n
= \underbrace{-4n(1-n^2)}_{\text{Volumen}} - \kappa\nabla^2 n
$$

Dies ist eine **PDE vierter Ordnung** im Raum.
FEniCS behandelt PDEs zweiter Ordnung auf natürliche Weise, daher zerlegen wir sie in zwei gekoppelte
Gleichungen zweiter Ordnung, indem wir das chemische Potential $\mu$ als unabhängige Variable einführen:

$$
\text{(1)}\quad \frac{n - n_\mathrm{old}}{\Delta t} = \nabla^2 \mu_\theta
\qquad
\text{(2)}\quad \mu = \frac{\partial G}{\partial n} - \kappa\nabla^2 n
$$

wobei $\mu_\theta = \theta\,\mu + (1-\theta)\,\mu_\mathrm{old}$ ein
Crank-Nicholson-Zeitmittel ist ($\theta = 0.5$).  
Wir lösen für $n$ und $\mu$ **gleichzeitig** mit einem gemischten Finite-Elemente-Raum.

In [ ]:
# ================================================================
#  TEIL 2 PARAMETER  — ändern Sie diese, um die Phasenseparation zu erkunden
# ================================================================

# --- Physikalische Parameter ---
kappa  = 0.01   # Grenzflächenenergiekoeffizient (steuert Grenzflächenbreite)
                # AKTION: probieren Sie 0.001, 0.005, 0.01, 0.05

# --- Anfangsbedingungen ---
n_mean = 0.0    # Mittlere Zusammensetzung  (-1 < n_mean < 1)
                # AKTION: probieren Sie -0.3, 0.0, 0.3
noise  = 0.05   # Amplitude der anfänglichen zufälligen Fluktuationen (klein halten)

# --- Numerische Parameter ---
nx_ch  = 64     # Gitterunterteilungen pro Seite  (größer = mehr Detail, langsamer)
ny_ch  = 64
dt     = 1.0e-4 # Zeitschritt  — verringern, wenn der Löser nicht konvergiert
                # AKTION: probieren Sie 5e-5 oder 5e-6
theta  = 0.5    # 0.5 = Crank-Nicholson,  1.0 = Rückwärts-Euler
                # AKTION: probieren Sie 1.0, um die Stabilität zu vergleichen

# --- Simulationslänge ---
n_steps    = 1000  # Gesamtzahl der Zeitschritte
                   # AKTION: erhöhen, um Vergröberung im Spätstadium zu beobachten
save_every = 25    # Alle diese Schritte einen Schnappschuss speichern

print(f"kappa={kappa}  n_mean={n_mean}  noise={noise}")
print(f"Gitter {nx_ch}x{ny_ch}  dt={dt}  theta={theta}")
print(f"{n_steps} Schritte, Schnappschuss alle {save_every}")

In [ ]:
# --- Gitter ---
mesh_ch = UnitSquareMesh(nx_ch, ny_ch)

# --- Gemischter Funktionsraum ---
# Wir lösen für (n, mu) gleichzeitig.
# Jede Komponente verwendet stückweise lineare (P1) Lagrange-Elemente.
P1 = FiniteElement("Lagrange", mesh_ch.ufl_cell(), 1)
ME = FunctionSpace(mesh_ch, P1 * P1)   # Produktraum: (n, mu)

print(f"Gemischter FE-Raum:  {ME.dim()} DOFs gesamt  ({(nx_ch+1)*(ny_ch+1)} pro Komponente)")

# Ansatz- und Testfunktionen für den gemischten Raum
du   = TrialFunction(ME)       # Inkrement für Newton-Linearisierung
q, v = TestFunctions(ME)       # Testfunktionen für Gleichungen (1) und (2)

# Lösungsfunktionen: aktueller und vorheriger Zeitschritt
u_new = Function(ME)   # (n, mu) zum neuen Zeitschritt
u_old = Function(ME)   # (n, mu) zum vorherigen Zeitschritt

# Symbolische Aufteilung — liefert UFL-Teilausdrücke für die schwache Form
n,  mu  = split(u_new)
n0, mu0 = split(u_old)

# --- Anfangsbedingungen ---
# Start aus einer nahezu gleichmäßigen Mischung (n = n_mean) mit kleinem zufälligem Rauschen.
# Ohne Rauschen würde der perfekt gleichmäßige Zustand seine Symmetrie nie brechen.
class InitialConditions(UserExpression):
    def __init__(self, mean, amplitude, seed=42, **kwargs):
        super().__init__(**kwargs)
        self.mean = mean
        self.amplitude = amplitude
        random.seed(seed)

    def eval(self, values, x):
        values[0] = self.mean + self.amplitude * (2*random.random() - 1)  # n
        values[1] = 0.0                                                     # mu

    def value_shape(self):
        return (2,)   # zweikomponentige Ausgabe

u_init = InitialConditions(mean=n_mean, amplitude=noise, degree=1)
u_new.interpolate(u_init)
u_old.interpolate(u_init)

# Schnelle Plausibilitätsprüfung des Anfangsfeldes
n_func_init, _ = u_new.split(deepcopy=True)
print(f"Anfangs-n-Bereich: [{n_func_init.vector().min():.3f}, {n_func_init.vector().max():.3f}]")

In [ ]:
# --- Freie Energie und chemisches Potential via automatische UFL-Differentiation ---

# n als UFL-Variable markieren, damit diff() danach differenzieren kann
n_var = variable(n)

# Doppelmulden-Volumenfreie Energie (aus Vorlesung):  G(n) = (1 - n^2)^2
G    = (1 - n_var**2)**2
dGdn = diff(G, n_var)       # automatische Differentiation: dG/dn = -4n(1-n^2)

# Crank-Nicholson-Zeitmittel des chemischen Potentials
mu_mid = theta * mu + (1 - theta) * mu0

# --- Schwache Form (Residuum F = 0) ---
#
# Gleichung (1) — Massenerhaltung:
#   Integral[ (n-n0)/dt * q ] dx  +  Integral[ grad(mu_theta).grad(q) ] dx  = 0
F0 = ((n - n0) / dt) * q * dx  +  dot(grad(mu_mid), grad(q)) * dx

# Gleichung (2) — Definition des chemischen Potentials:
#   Integral[ mu * v ] dx  -  Integral[ dG/dn * v ] dx  -  kappa * Integral[ grad(n).grad(v) ] dx  = 0
F1 = mu * v * dx  -  dGdn * v * dx  -  kappa * dot(grad(n), grad(v)) * dx

F = F0 + F1   # kombiniertes Residuum

# --- Jacobi-Matrix (für das Newton-Verfahren erforderlich) ---
# UFL berechnet die Richtungsableitung von F bezüglich u_new automatisch.
J = derivative(F, u_new, du)

# --- Löser-Konfiguration ---
# Keine Dirichlet-RB: Null-Fluss-Neumann-Bedingungen sind die natürlichen/Standard-RB
problem = NonlinearVariationalProblem(F, u_new, J=J)
solver  = NonlinearVariationalSolver(problem)

prm = solver.parameters["newton_solver"]
prm["linear_solver"]       = "lu"   # direkter Löser, zuverlässig bis ~100x100 Gitter
prm["relative_tolerance"]  = 1e-6
prm["maximum_iterations"]  = 15

print("Schwache Form assembliert.")
print("Löser konfiguriert  (Newton + LU).")

In [ ]:
# --- Hilfsfunktion: Zusammensetzungsfeld und freie Energie zu einem gegebenen Zeitpunkt extrahieren ---
def save_snapshot(u_now, t_now):
    """Zusammensetzungsgitter und Energiewerte für den Zeitpunkt t_now anhängen."""
    n_f, _ = u_now.split(deepcopy=True)

    # Knotenwerte extrahieren und das regelmäßige Gitter rekonstruieren.
    # Für UnitSquareMesh(nx, ny) liegen Knoten an Positionen (i/nx, j/ny).
    # Sortierung nach (y, x) stellt das zeilenweise Gitter wieder her.
    coords   = mesh_ch.coordinates()                  # Form: (n_verts, 2)
    vtx_vals = n_f.compute_vertex_values(mesh_ch)     # Werte an jedem Knoten
    idx      = np.lexsort((coords[:, 0], coords[:, 1]))
    Z        = vtx_vals[idx].reshape(ny_ch + 1, nx_ch + 1)

    # Freienergieintegrale mit FEniCS berechnen
    E_bulk  = float(assemble((1 - n_f**2)**2 * dx))
    E_iface = float(assemble(0.5 * kappa * dot(grad(n_f), grad(n_f)) * dx))

    snapshots.append(Z.copy())
    snap_times.append(t_now)
    energies.append((t_now, E_bulk, E_iface))


# Speicher initialisieren
snapshots  = []
snap_times = []
energies   = []

save_snapshot(u_new, 0.0)   # Anfangszustand speichern

# --- Zeitschrittschleife ---
print("Cahn-Hilliard-Simulation wird gestartet...")
t_sim = 0.0
t0    = timer.time()

for step in range(1, n_steps + 1):

    # Aktuelle Lösung vor dem Weiterrechnen archivieren
    u_old.assign(u_new)

    # Nichtlineares System für den nächsten Zeitschritt lösen
    solver.solve()
    t_sim += dt

    if step % save_every == 0:
        save_snapshot(u_new, t_sim)
        E_tot   = energies[-1][1] + energies[-1][2]
        elapsed = timer.time() - t0
        print(f"  Schritt {step:4d}/{n_steps}   t={t_sim:.2e}   E={E_tot:.4f}   [{elapsed:.0f}s]")

print(f"\nFertig. {len(snapshots)} Schnappschüsse gespeichert.")

In [ ]:
# --- Zusammensetzungs-Schnappschüsse visualisieren ---
n_snaps = len(snapshots)
cols    = min(n_snaps, 5)
rows    = (n_snaps + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(3.5 * cols, 3.5 * rows + 0.5),
                         squeeze=False)
axes = axes.flatten()

for i, (Z, t_val) in enumerate(zip(snapshots, snap_times)):
    im = axes[i].imshow(
        Z, origin="lower", vmin=-1, vmax=1,
        cmap="RdBu_r", extent=[0, 1, 0, 1], interpolation="bilinear"
    )
    axes[i].set_title(f"t = {t_val:.1e}", fontsize=10)
    axes[i].set_xlabel("x", fontsize=9)
    if i % cols == 0:
        axes[i].set_ylabel("y", fontsize=9)

for i in range(n_snaps, len(axes)):
    axes[i].set_visible(False)

fig.subplots_adjust(right=0.88, hspace=0.45, wspace=0.3)
cbar_ax = fig.add_axes([0.90, 0.15, 0.02, 0.7])
cb = fig.colorbar(im, cax=cbar_ax)
cb.set_label("Zusammensetzung n", fontsize=10)
cb.set_ticks([-1, 0, 1])
cb.set_ticklabels(["-1 (Phase A)", "0", "+1 (Phase B)"])

fig.suptitle(f"Spinodale Entmischung  (kappa={kappa}, n_mean={n_mean})",
             fontsize=12, y=1.02)
plt.show()

In [ ]:
# --- Freienergieverlauf plotten ---
times_e  = [e[0] for e in energies]
E_bulk_e = [e[1] for e in energies]
E_iface_e= [e[2] for e in energies]
E_total_e= [b + g_e for b, g_e in zip(E_bulk_e, E_iface_e)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Energiekomponenten
axes[0].plot(times_e, E_total_e, "k-o",  label="Gesamt E",              linewidth=2, markersize=5)
axes[0].plot(times_e, E_bulk_e,  "b--s", label="Volumenenergie G(n)",   linewidth=2, markersize=5)
axes[0].plot(times_e, E_iface_e, "r--^", label="Grenzflächenenergie",   linewidth=2, markersize=5)
axes[0].set_xlabel("Zeit t")
axes[0].set_ylabel("Freie Energie E")
axes[0].set_title("Freienergiekomponenten")
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Relativer Abfall
E0  = E_total_e[0]
pct = [(E - E0) / abs(E0) * 100 for E in E_total_e]
axes[1].plot(times_e, pct, "k-o", linewidth=2, markersize=5)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_xlabel("Zeit t")
axes[1].set_ylabel("Relative Änderung (%)")
axes[1].set_title("Abnahme der freien Gesamtenergie")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

drop = (E_total_e[-1] - E_total_e[0]) / abs(E_total_e[0]) * 100
print(f"Freie Energie nahm um {drop:.1f}% über {n_steps} Schritte ab.")
print("Das System entwickelt sich spontan zu niedrigerer freier Energie, wie die Thermodynamik verlangt.")

### Aufgabe 2: Spinodale Entmischung erkunden

Kehren Sie zur **Parameter**-Zelle oben zurück und probieren Sie die folgenden Experimente aus.
Führen Sie bei jeder Änderung ab der Parameter-Zelle neu aus.

**2.1 — Grenzflächenenergie $\kappa$**  
Probieren Sie `kappa = 0.001, 0.005, 0.01, 0.05`.
- Wie ändert sich die charakteristische Größe der phasenseparierten Domänen mit $\kappa$?
  (*Hinweis: Die spinodale Wellenlänge skaliert als $\lambda \propto \sqrt{\kappa}$.)
- Führt ein kleineres $\kappa$ zu feinerem oder gröberem Gefüge?

**2.2 — Anfangszusammensetzung $\bar{n}_0$**  
Probieren Sie `n_mean = 0.0, 0.2, 0.4`.
- Beschreiben Sie die Morphologie: Sehen Sie ein zusammenhängendes Netzwerk oder isolierte Tröpfchen
  einer Phase in einer Matrix der anderen?
- Der Übergang zwischen diesen Morphologien ist der **Perkolationsübergang**.

**2.3 — Vergröberung im Spätstadium**  
Setzen Sie `n_steps = 1000` (bei $\kappa = 0.001$, `n_mean = 0.0`).
- Beobachten Sie, wie Domänen über die Zeit **vergröbern** (verschmelzen) — dies ist **Ostwald-Reifung**.
- Nimmt die freie Energie durchgehend ab?

**2.4 — Zeitintegrationsschema**  
Probieren Sie `theta = 1.0` (Rückwärts-Euler) und einen größeren Zeitschritt `dt = 5e-5`.
- Ist das vollständig implizite Schema stabiler oder weniger stabil als Crank-Nicholson?
- Was passiert, wenn Sie `dt = 1e-4` versuchen?

---
## Zusammenfassung

In diesem Lab haben Sie:

- **Die Poisson-Gleichung gelöst** für die Membrandurchbiegung mit FEM in FEniCS
- **Gitterkonvergenz verifiziert** und den Rechenaufwand der Verfeinerung verstanden
- **Spinodale Entmischung simuliert** mit dem Cahn-Hilliard-Phasenfeldmodell
- **Beobachtet**, wie $\kappa$ die Gefügelängenskala steuert und wie asymmetrische
  Zusammensetzungen zu qualitativ unterschiedlichen Morphologien führen
- **Die freie Energie verfolgt**, die sich bergab in Richtung thermodynamischem Gleichgewicht entwickelt

| Modell | Gleichung | Schlüsselparameter | Anwendungsbeispiel |
|---|---|---|---|
| Poisson | $-\nabla^2 u = g$ | Lastverteilung $g$ | Elastizität, Wärme, Diffusion |
| Cahn-Hilliard | $\partial_t n = \nabla^2(\partial G/\partial n - \kappa\nabla^2 n)$ | Grenzflächenenergie $\kappa$ | Phasenseparation, Gefüge |

Beide werden mit demselben Werkzeug gelöst — der **Finite-Elemente-Methode** — die
kontinuierliche PDEs in endliche algebraische Systeme umwandelt, sobald man die schwache Form bereitstellt.

### Weiterführende Literatur

- **FEniCS-Tutorial**: *Solving PDEs in Python* — Langtangen & Logg (kostenlos online unter [fenicsproject.org](https://fenicsproject.org/pub/tutorial/html/))
- **Originalartikel**: Cahn & Hilliard, *J. Chem. Phys.* **28**(2), 258 (1958)
- **Phasenfeldübersicht**: Moelans, Blanpain & Wollants, *CALPHAD* **32**(2), 268 (2008)